In [1]:
# ============================================================
# CELL 1 — CREATE PAYMENT TRANSACTION DATA
# ============================================================
#
# WHY ARE WE DOING THIS?
#
# Our AI needs historical payment transactions to learn from.
# Each transaction contains information such as:
#
# amount, payment method, bank, previous failures, etc.
#
# We are creating 20,000 example transactions for our prototype.
# ============================================================

import pandas as pd
import numpy as np

# This makes our random data reproducible.
# If we run the code again, we get the same type of data.
np.random.seed(42)

# Number of transactions we want
n = 20000


# ---------------- BASIC TRANSACTION INFORMATION ----------------

# Unique ID for every transaction
transaction_id = [
    f"TX{i:05d}" for i in range(1, n + 1)
]

# Transaction amount between ₹100 and ₹20,000
amount = np.random.randint(100, 20000, n)

# Payment method used by customer
payment_method = np.random.choice(
    ["UPI", "CARD", "NET_BANKING"],
    n,
    p=[0.55, 0.30, 0.15]
)

# Bank involved in the transaction
bank = np.random.choice(
    ["Bank_A", "Bank_B", "Bank_C", "Bank_D"],
    n,
    p=[0.30, 0.25, 0.25, 0.20]
)

# Device used by customer
device = np.random.choice(
    ["Android", "iOS", "Desktop"],
    n,
    p=[0.60, 0.20, 0.20]
)

# Hour at which payment was attempted
hour = np.random.randint(0, 24, n)

# Type of customer
customer_type = np.random.choice(
    ["NEW", "RETURNING"],
    n,
    p=[0.40, 0.60]
)


# ---------------- CUSTOMER PAYMENT HISTORY ----------------

# Number of previous failed payments
previous_failures = np.random.choice(
    [0, 1, 2, 3],
    n,
    p=[0.65, 0.20, 0.10, 0.05]
)

# Current attempt number
attempt_number = previous_failures + 1


# ---------------- CREATE PAYMENT RISK ----------------

# Start with zero risk for every transaction
risk = np.zeros(n)

# More previous failures → higher risk
risk += previous_failures * 0.18

# Multiple attempts → higher risk
risk += np.where(
    attempt_number >= 3,
    0.15,
    0
)

# Some banks have slightly higher simulated risk
risk += np.where(bank == "Bank_C", 0.08, 0)
risk += np.where(bank == "Bank_D", 0.05, 0)

# Peak hours: 6 PM to 10 PM
peak_hours = (hour >= 18) & (hour <= 22)

risk += np.where(
    peak_hours,
    0.08,
    0
)

# Payment method effect
risk += np.where(payment_method == "CARD", 0.03, 0)
risk += np.where(payment_method == "NET_BANKING", 0.05, 0)

# High-value transaction
risk += np.where(
    amount > 15000,
    0.05,
    0
)

# New customer
risk += np.where(
    customer_type == "NEW",
    0.03,
    0
)


# ---------------- CREATE ERROR CONDITIONS ----------------

error_code = []

for i in range(n):

    # Higher risk → higher chance of an error
    probability = min(
        0.15 + risk[i],
        0.85
    )

    if np.random.random() < probability:

        if payment_method[i] == "CARD":

            error = np.random.choice(
                [
                    "CARD_DECLINED",
                    "NETWORK_ERROR",
                    "AUTH_ERROR"
                ],
                p=[0.50, 0.25, 0.25]
            )

        elif payment_method[i] == "UPI":

            error = np.random.choice(
                [
                    "BANK_ERROR",
                    "NETWORK_ERROR",
                    "AUTH_ERROR"
                ],
                p=[0.50, 0.30, 0.20]
            )

        else:

            error = np.random.choice(
                [
                    "BANK_ERROR",
                    "NETWORK_ERROR",
                    "AUTH_ERROR"
                ],
                p=[0.45, 0.35, 0.20]
            )

    else:

        error = "NONE"

    error_code.append(error)


# ---------------- CREATE FINAL PAYMENT RESULT ----------------

status = []

for i in range(n):

    final_risk = risk[i]

    # If an error occurred, failure probability increases
    if error_code[i] != "NONE":
        final_risk += 0.30

    # Convert risk into failure probability
    failure_probability = min(
        0.10 + final_risk,
        0.95
    )

    # Decide SUCCESS or FAILED
    if np.random.random() < failure_probability:
        status.append("FAILED")
    else:
        status.append("SUCCESS")


# ---------------- CREATE FINAL DATAFRAME ----------------

df = pd.DataFrame({

    "transaction_id": transaction_id,
    "amount": amount,
    "payment_method": payment_method,
    "bank": bank,
    "device": device,
    "hour": hour,
    "previous_failures": previous_failures,
    "attempt_number": attempt_number,
    "error_code": error_code,
    "customer_type": customer_type,
    "status": status
})


# ---------------- CHECK DATASET ----------------

print("✅ Dataset created successfully!")

print("\nDataset shape:")
print(df.shape)

print("\nPayment results:")
print(df["status"].value_counts())

display(df.head())

✅ Dataset created successfully!

Dataset shape:
(20000, 11)

Payment results:
status
SUCCESS    12151
FAILED      7849
Name: count, dtype: int64


,transaction_id,amount,payment_method,bank,device,hour,previous_failures,attempt_number,error_code,customer_type,status
0,TX00001,15895,CARD,Bank_D,Android,9,0,1,CARD_DECLINED,RETURNING,SUCCESS
1,TX00002,960,UPI,Bank_A,Android,20,0,1,NONE,RETURNING,SUCCESS
2,TX00003,5490,CARD,Bank_D,Desktop,3,0,1,NONE,RETURNING,SUCCESS
3,TX00004,12064,UPI,Bank_B,Android,23,0,1,NONE,NEW,SUCCESS
4,TX00005,11384,NET_BANKING,Bank_C,Desktop,20,0,1,NONE,RETURNING,SUCCESS


In [2]:
# ============================================================
# CELL 2 — SAVE OUR DATASET
# ============================================================
#
# WHY?
#
# We save our transaction data as a CSV file.
# Later our ML model will read this file.
# ============================================================

df.to_csv(
    "payment_transactions_realistic.csv",
    index=False
)

print("✅ Dataset saved successfully!")

✅ Dataset saved successfully!


In [3]:
# ============================================================
# CELL 3 — LOOK AT OUR DATA
# ============================================================
#
# WHY?
#
# Before teaching AI, we should understand what data
# we are giving to it.
# ============================================================

print("Number of transactions:", len(df))

print("\nColumns:")
print(df.columns.tolist())

print("\nPayment result:")
print(df["status"].value_counts())

print("\nFirst 5 transactions:")
display(df.head())

Number of transactions: 20000

Columns:
['transaction_id', 'amount', 'payment_method', 'bank', 'device', 'hour', 'previous_failures', 'attempt_number', 'error_code', 'customer_type', 'status']

Payment result:
status
SUCCESS    12151
FAILED      7849
Name: count, dtype: int64

First 5 transactions:


,transaction_id,amount,payment_method,bank,device,hour,previous_failures,attempt_number,error_code,customer_type,status
0,TX00001,15895,CARD,Bank_D,Android,9,0,1,CARD_DECLINED,RETURNING,SUCCESS
1,TX00002,960,UPI,Bank_A,Android,20,0,1,NONE,RETURNING,SUCCESS
2,TX00003,5490,CARD,Bank_D,Desktop,3,0,1,NONE,RETURNING,SUCCESS
3,TX00004,12064,UPI,Bank_B,Android,23,0,1,NONE,NEW,SUCCESS
4,TX00005,11384,NET_BANKING,Bank_C,Desktop,20,0,1,NONE,RETURNING,SUCCESS


In [4]:
# ============================================================
# STEP 1 — TRAIN REAL PAYMENT FAILURE RISK MODEL
# ============================================================

import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score


# ------------------------------------------------------------
# 1. USE THE EXISTING DATASET
# ------------------------------------------------------------

# If the dataset variable already exists, use it.
# Otherwise load the saved CSV.

if "df" not in globals():

    import glob

    csv_files = glob.glob("*.csv")

    print("CSV files found:")
    print(csv_files)

    # Use the first CSV containing our payment dataset columns
    for file in csv_files:

        temp = pd.read_csv(file)

        required_columns = {
            "transaction_id",
            "amount",
            "payment_method",
            "bank",
            "device",
            "hour",
            "previous_failures",
            "attempt_number",
            "error_code",
            "customer_type",
            "status"
        }

        if required_columns.issubset(temp.columns):

            df = temp.copy()

            print("\n✅ Existing payment dataset loaded:")
            print(file)

            break


# ------------------------------------------------------------
# 2. VERIFY DATASET
# ------------------------------------------------------------

print("\n========== DATASET CHECK ==========")

print("Dataset shape:", df.shape)

print("\nTarget distribution:")
print(df["status"].value_counts())


# ------------------------------------------------------------
# 3. FEATURES AND TARGET
# ------------------------------------------------------------

features = [
    "amount",
    "payment_method",
    "bank",
    "device",
    "hour",
    "previous_failures",
    "attempt_number",
    "error_code",
    "customer_type"
]

target = "status"


X = df[features]

y = df[target]


# ------------------------------------------------------------
# 4. IDENTIFY COLUMN TYPES
# ------------------------------------------------------------

categorical_features = [
    "payment_method",
    "bank",
    "device",
    "error_code",
    "customer_type"
]

numeric_features = [
    "amount",
    "hour",
    "previous_failures",
    "attempt_number"
]


# ------------------------------------------------------------
# 5. PREPROCESSING
# ------------------------------------------------------------

preprocessor = ColumnTransformer(

    transformers=[

        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),

        (
            "numeric",
            "passthrough",
            numeric_features
        )

    ]
)


# ------------------------------------------------------------
# 6. REAL ML CLASSIFIER
# ------------------------------------------------------------

clf = RandomForestClassifier(

    n_estimators=200,

    random_state=42,

    class_weight="balanced",

    n_jobs=-1

)


# ------------------------------------------------------------
# 7. COMPLETE ML PIPELINE
# ------------------------------------------------------------

pipeline = Pipeline(

    steps=[

        (
            "preprocessor",
            preprocessor
        ),

        (
            "classifier",
            clf
        )

    ]

)


# ------------------------------------------------------------
# 8. TRAIN / TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)


# ------------------------------------------------------------
# 9. TRAIN MODEL
# ------------------------------------------------------------

print("\n========== TRAINING MODEL ==========")

pipeline.fit(
    X_train,
    y_train
)

print("✅ Model training completed!")


# ------------------------------------------------------------
# 10. TEST MODEL
# ------------------------------------------------------------

y_pred = pipeline.predict(X_test)

accuracy = accuracy_score(
    y_test,
    y_pred
)

print("\n========== MODEL PERFORMANCE ==========")

print(
    f"Test Accuracy: {accuracy * 100:.2f}%"
)

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred
    )
)


# ------------------------------------------------------------
# 11. CHECK MODEL CLASSES
# ------------------------------------------------------------

print("\n========== MODEL CLASSES ==========")

print(
    pipeline.classes_
)


# ------------------------------------------------------------
# 12. SAVE REAL TRAINED MODEL
# ------------------------------------------------------------

joblib.dump(
    pipeline,
    "payment_model.pkl"
)

print(
    "\n✅ REAL ML MODEL SAVED SUCCESSFULLY!"
)

print(
    "File: payment_model.pkl"
)

print(
    "This model predicts SUCCESS / FAILED payment risk."
)


========== DATASET CHECK ==========
Dataset shape: (20000, 11)

Target distribution:
status
SUCCESS    12151
FAILED      7849
Name: count, dtype: int64

========== TRAINING MODEL ==========
✅ Model training completed!

========== MODEL PERFORMANCE ==========
Test Accuracy: 70.58%

Classification Report:
              precision    recall  f1-score   support

      FAILED       0.64      0.57      0.61      1570
     SUCCESS       0.74      0.79      0.77      2430

    accuracy                           0.71      4000
   macro avg       0.69      0.68      0.69      4000
weighted avg       0.70      0.71      0.70      4000


========== MODEL CLASSES ==========
['FAILED' 'SUCCESS']

✅ REAL ML MODEL SAVED SUCCESSFULLY!
File: payment_model.pkl
This model predicts SUCCESS / FAILED payment risk.


In [5]:
# ============================================================
# MODEL VERIFICATION — RUN AFTER ML TRAINING CELL
# ============================================================

print("========== MODEL VERIFICATION ==========")

print("Model object exists:", "pipeline" in globals())

if "pipeline" in globals():

    print("Model classes:", pipeline.classes_)

    print("Training samples:", len(X_train))
    print("Testing samples:", len(X_test))

    print("\nModel is ready for payment risk prediction.")

else:

    print("❌ Model not found. Please check the training cell.")

========== MODEL VERIFICATION ==========
Model object exists: True
Model classes: ['FAILED' 'SUCCESS']
Training samples: 16000
Testing samples: 4000

Model is ready for payment risk prediction.


In [6]:
# ============================================================
# ADD AFTER MODEL VERIFICATION CELL
# REAL PAYMENT RISK PREDICTION
# ============================================================

print("========== REAL PAYMENT RISK PREDICTION ==========")

# Take one transaction from the existing test dataset
sample_payment = X_test.iloc[[0]]

# Get ML prediction probabilities
probabilities = pipeline.predict_proba(sample_payment)

# Find FAILED class
failed_index = list(pipeline.classes_).index("FAILED")

# Calculate failure probability
failure_probability = probabilities[0][failed_index] * 100

# Convert probability into risk level
if failure_probability < 30:
    risk_level = "LOW"
elif failure_probability < 70:
    risk_level = "MEDIUM"
else:
    risk_level = "HIGH"

print("Payment details:")
print(sample_payment)

print("\nAI Failure Probability:")
print(f"{failure_probability:.2f}%")

print("\nAI Risk Level:")
print(risk_level)

========== REAL PAYMENT RISK PREDICTION ==========
Payment details:
       amount payment_method    bank   device  hour  previous_failures  \
13028    9508            UPI  Bank_A  Desktop    17                  1   

       attempt_number error_code customer_type  
13028               2       NONE     RETURNING  

AI Failure Probability:
25.50%

AI Risk Level:
LOW


In [7]:
# ============================================================
# REAL FAILED PAYMENT → AI RISK PREDICTION
# ADD AFTER REAL PAYMENT RISK PREDICTION CELL
# BEFORE ORIGINAL CELL 4
# ============================================================

print("========== FAILED PAYMENT RISK PREDICTION ==========")

# ------------------------------------------------------------
# 1. Find a FAILED transaction from the existing test dataset
# ------------------------------------------------------------

failed_test_indices = y_test[y_test == "FAILED"].index

# Select the first actual FAILED transaction
failed_index = failed_test_indices[0]

failed_payment = X_test.loc[[failed_index]]


# ------------------------------------------------------------
# 2. Ask the REAL ML model for failure probability
# ------------------------------------------------------------

probabilities = pipeline.predict_proba(
    failed_payment
)


# ------------------------------------------------------------
# 3. Find FAILED class
# ------------------------------------------------------------

failed_class_index = list(
    pipeline.classes_
).index("FAILED")


# ------------------------------------------------------------
# 4. Calculate failure probability
# ------------------------------------------------------------

failure_probability = (
    probabilities[0][failed_class_index] * 100
)


# ------------------------------------------------------------
# 5. Determine risk level
# ------------------------------------------------------------

if failure_probability < 30:

    risk_level = "LOW"

elif failure_probability < 70:

    risk_level = "MEDIUM"

else:

    risk_level = "HIGH"


# ------------------------------------------------------------
# 6. Display result
# ------------------------------------------------------------

print("\nActual transaction status:")
print("FAILED")

print("\nPayment details:")
print(failed_payment)

print("\nAI Failure Probability:")
print(f"{failure_probability:.2f}%")

print("\nAI Risk Level:")
print(risk_level)

print("\n✅ REAL FAILED PAYMENT IDENTIFIED AND ANALYZED")

========== FAILED PAYMENT RISK PREDICTION ==========

Actual transaction status:
FAILED

Payment details:
       amount payment_method    bank   device  hour  previous_failures  \
16042    7974    NET_BANKING  Bank_D  Desktop     1                  2   

       attempt_number  error_code customer_type  
16042               3  AUTH_ERROR     RETURNING  

AI Failure Probability:
92.50%

AI Risk Level:
HIGH

✅ REAL FAILED PAYMENT IDENTIFIED AND ANALYZED


In [8]:
# ============================================================
# FAILED PAYMENT → AI RECOVERY ACTION
# ADD AFTER FAILED PAYMENT RISK PREDICTION CELL
# BEFORE ORIGINAL CELL 4
# ============================================================

print("========== AI RECOVERY ACTION ==========")

# ------------------------------------------------------------
# Use the failed payment already selected above
# ------------------------------------------------------------

print("Payment Status: FAILED")
print(f"Failure Probability: {failure_probability:.2f}%")
print(f"Risk Level: {risk_level}")

# ------------------------------------------------------------
# Read the payment error from the failed transaction
# ------------------------------------------------------------

failed_error = failed_payment["error_code"].iloc[0]

print(f"Payment Error: {failed_error}")


# ------------------------------------------------------------
# AI recovery recommendation
# ------------------------------------------------------------

if failed_error == "AUTH_ERROR":

    recovery_action = "Complete Authentication & Retry"

elif failed_error == "CARD_DECLINED":

    recovery_action = "Try Alternative Payment Method"

elif failed_error == "BANK_ERROR":

    recovery_action = "Try Another Bank or Payment Method"

elif failed_error == "NETWORK_ERROR":

    recovery_action = "Retry Payment After Short Delay"

elif risk_level == "HIGH":

    recovery_action = "Try Alternative Payment Method"

elif risk_level == "MEDIUM":

    recovery_action = "Continue Payment or Use Another Method"

else:

    recovery_action = "No Recovery Action Required"


# ------------------------------------------------------------
# Display recommendation
# ------------------------------------------------------------

print("\n🧠 AI Recommended Recovery Action:")
print(recovery_action)

print("\n✅ Recovery action generated from the payment risk/error.")

========== AI RECOVERY ACTION ==========
Payment Status: FAILED
Failure Probability: 92.50%
Risk Level: HIGH
Payment Error: AUTH_ERROR

🧠 AI Recommended Recovery Action:
Complete Authentication & Retry

✅ Recovery action generated from the payment risk/error.


In [9]:
# ============================================================
# RECOVERY SIMULATION → RECOVERED / UNRECOVERED AMOUNT
# CORRECTED VERSION
# ============================================================

import random

print("========== RECOVERY SIMULATION ==========")

# ------------------------------------------------------------
# Payment amount from the real failed transaction
# ------------------------------------------------------------

recovery_amount = failed_payment["amount"].iloc[0]

print(f"At-Risk Payment Amount: ₹{recovery_amount:,.2f}")
print(f"Recovery Action: {recovery_action}")


# ------------------------------------------------------------
# Simulate recovery
# ------------------------------------------------------------

recovery_probability = 0.70

random_value = random.random()

recovery_success = random_value < recovery_probability


# ------------------------------------------------------------
# Calculate recovered / unrecovered amount
# ------------------------------------------------------------

if recovery_success:

    recovered_amount = recovery_amount
    unrecovered_amount = 0

else:

    recovered_amount = 0
    unrecovered_amount = recovery_amount


# ------------------------------------------------------------
# Display result
# ------------------------------------------------------------

print("\n========== RECOVERY RESULT ==========")

if recovery_success:

    print("✅ Recovery Successful")
    print(f"Recovered Amount: ₹{recovered_amount:,.2f}")
    print(f"Unrecovered Amount: ₹{unrecovered_amount:,.2f}")

else:

    print("❌ Recovery Failed")
    print(f"Recovered Amount: ₹{recovered_amount:,.2f}")
    print(f"Unrecovered Amount: ₹{unrecovered_amount:,.2f}")


print("\n✅ Recovery simulation completed.")

========== RECOVERY SIMULATION ==========
At-Risk Payment Amount: ₹7,974.00
Recovery Action: Complete Authentication & Retry

========== RECOVERY RESULT ==========
❌ Recovery Failed
Recovered Amount: ₹0.00
Unrecovered Amount: ₹7,974.00

✅ Recovery simulation completed.


In [10]:
# ============================================================
# RECOVERY METRICS
# ADD AFTER RECOVERY SIMULATION CELL
# BEFORE ORIGINAL CELL 4
# ============================================================

print("========== RECOVERY METRICS ==========")

# ------------------------------------------------------------
# Initialize metrics
# ------------------------------------------------------------

total_at_risk_amount = recovery_amount

total_recovered_amount = recovered_amount

total_unrecovered_amount = unrecovered_amount

total_recovery_attempts = 1

successful_recoveries = 1 if recovery_success else 0


# ------------------------------------------------------------
# Calculate recovery success rate
# ------------------------------------------------------------

recovery_success_rate = (
    successful_recoveries
    / total_recovery_attempts
) * 100


# ------------------------------------------------------------
# Display business metrics
# ------------------------------------------------------------

print(f"Total At-Risk Amount: ₹{total_at_risk_amount:,.2f}")

print(
    f"Total Recovered Amount: "
    f"₹{total_recovered_amount:,.2f}"
)

print(
    f"Total Unrecovered Amount: "
    f"₹{total_unrecovered_amount:,.2f}"
)

print(
    f"Total Recovery Attempts: "
    f"{total_recovery_attempts}"
)

print(
    f"Successful Recoveries: "
    f"{successful_recoveries}"
)

print(
    f"Recovery Success Rate: "
    f"{recovery_success_rate:.1f}%"
)

print("\n✅ Recovery metrics calculated.")


========== RECOVERY METRICS ==========
Total At-Risk Amount: ₹7,974.00
Total Recovered Amount: ₹0.00
Total Unrecovered Amount: ₹7,974.00
Total Recovery Attempts: 1
Successful Recoveries: 0
Recovery Success Rate: 0.0%

✅ Recovery metrics calculated.


In [11]:
# ============================================================
# AUDIT LOG
# ADD AFTER RECOVERY METRICS CELL
# BEFORE ORIGINAL CELL 4
# ============================================================

print("========== AUDIT LOG ==========")

# ------------------------------------------------------------
# Create audit record for this recovery decision
# ------------------------------------------------------------

audit_record = {
    "Transaction ID": (
        failed_payment["transaction_id"].iloc[0]
        if "transaction_id" in failed_payment.columns
        else str(failed_index)
    ),
    "Payment Amount (₹)": recovery_amount,
    "Actual Status": "FAILED",
    "Failure Probability (%)": round(
        failure_probability, 2
    ),
    "Risk Level": risk_level,
    "Payment Error": failed_error,
    "Recovery Action": recovery_action,
    "Recovery Result": (
        "SUCCESS"
        if recovery_success
        else "FAILED"
    ),
    "Recovered Amount (₹)": recovered_amount,
    "Unrecovered Amount (₹)": unrecovered_amount
}


# ------------------------------------------------------------
# Convert audit record to DataFrame
# ------------------------------------------------------------

audit_log = pd.DataFrame(
    [audit_record]
)


# ------------------------------------------------------------
# Display audit log
# ------------------------------------------------------------

print("\nAI Decision Audit Record:\n")

display(audit_log)

print("\n✅ Audit log created successfully.")

========== AUDIT LOG ==========

AI Decision Audit Record:



,Transaction ID,Payment Amount (₹),Actual Status,Failure Probability (%),Risk Level,Payment Error,Recovery Action,Recovery Result,Recovered Amount (₹),Unrecovered Amount (₹)
0,16042,7974,FAILED,92.5,HIGH,AUTH_ERROR,Complete Authentication & Retry,FAILED,0,7974



✅ Audit log created successfully.


In [12]:
# ============================================================
# RECOVERY STOPPING RULE
# ADD AFTER AUDIT LOG CELL
# BEFORE ORIGINAL CELL 4
# ============================================================

print("========== RECOVERY STOPPING RULE ==========")

# ------------------------------------------------------------
# Maximum recovery attempts allowed
# ------------------------------------------------------------

MAX_RECOVERY_ATTEMPTS = 2


# ------------------------------------------------------------
# Current recovery attempt
# ------------------------------------------------------------

current_attempt = total_recovery_attempts


# ------------------------------------------------------------
# Apply stopping rule
# ------------------------------------------------------------

if recovery_success:

    stopping_decision = "STOP - PAYMENT RECOVERED"

elif current_attempt >= MAX_RECOVERY_ATTEMPTS:

    stopping_decision = (
        "STOP - MAXIMUM RECOVERY ATTEMPTS REACHED"
    )

else:

    stopping_decision = (
        "CONTINUE - TRY ANOTHER RECOVERY ACTION"
    )


# ------------------------------------------------------------
# Display stopping decision
# ------------------------------------------------------------

print(
    f"Current Recovery Attempt: "
    f"{current_attempt}"
)

print(
    f"Maximum Allowed Attempts: "
    f"{MAX_RECOVERY_ATTEMPTS}"
)

print(
    f"\nStopping Rule Decision:\n"
    f"{stopping_decision}"
)

print("\n✅ Stopping rule evaluated successfully.")

========== RECOVERY STOPPING RULE ==========
Current Recovery Attempt: 1
Maximum Allowed Attempts: 2

Stopping Rule Decision:
CONTINUE - TRY ANOTHER RECOVERY ACTION

✅ Stopping rule evaluated successfully.


In [13]:
# ============================================================
# SECOND RECOVERY ATTEMPT
# ============================================================

print("========== SECOND RECOVERY ATTEMPT ==========")

# Defaults prevent undefined-variable errors when the first
# recovery attempt succeeds and no second attempt is needed.
second_recovery_action = "N/A"
second_recovery_probability = 0.0
second_recovery_success = False
second_recovered_amount = 0
second_unrecovered_amount = recovery_amount

# ------------------------------------------------------------
# Check whether the stopping rule allows another attempt
# ------------------------------------------------------------

if stopping_decision == "CONTINUE - TRY ANOTHER RECOVERY ACTION":

    print("Recovery attempt 1 was unsuccessful.")
    print("Stopping rule allows another attempt.")

    # Select a different recovery action
    second_recovery_action = "Alternative Payment Method"

    print(
        f"\nSecond Recovery Action:\n"
        f"{second_recovery_action}"
    )

    # Simulate second recovery attempt
    random_value = random.random()
    second_recovery_probability = 0.70

    second_recovery_success = (
        random_value < second_recovery_probability
    )

    # Calculate second attempt amounts
    if second_recovery_success:
        second_recovered_amount = recovery_amount
        second_unrecovered_amount = 0
    else:
        second_recovered_amount = 0
        second_unrecovered_amount = recovery_amount

    print("\nSecond Recovery Result:")

    if second_recovery_success:
        print("✅ Recovery Successful")
    else:
        print("❌ Recovery Failed")

    print(
        f"Recovered Amount: ₹{second_recovered_amount:,.2f}"
    )

    print(
        f"Unrecovered Amount: ₹{second_unrecovered_amount:,.2f}"
    )

else:

    print("🛑 Recovery process stopped.")


========== SECOND RECOVERY ATTEMPT ==========
Recovery attempt 1 was unsuccessful.
Stopping rule allows another attempt.

Second Recovery Action:
Alternative Payment Method

Second Recovery Result:
✅ Recovery Successful
Recovered Amount: ₹7,974.00
Unrecovered Amount: ₹0.00


In [14]:
# ============================================================
# FINAL RECOVERY OUTCOME
# ============================================================

print("========== FINAL RECOVERY OUTCOME ==========")

final_at_risk_amount = recovery_amount
first_attempt_recovered = recovered_amount

# A second attempt is counted only if it was actually allowed.
if stopping_decision == "CONTINUE - TRY ANOTHER RECOVERY ACTION":

    if second_recovery_success:
        final_recovered_amount = second_recovered_amount
    else:
        final_recovered_amount = first_attempt_recovered

else:
    final_recovered_amount = first_attempt_recovered

# Each payment amount can be recovered only once.
final_recovered_amount = min(
    final_recovered_amount,
    final_at_risk_amount
)

final_unrecovered_amount = (
    final_at_risk_amount - final_recovered_amount
)

if final_recovered_amount >= final_at_risk_amount:
    final_status = "RECOVERED"
elif final_recovered_amount > 0:
    final_status = "PARTIALLY RECOVERED"
else:
    final_status = "UNRECOVERED"

if final_at_risk_amount > 0:
    final_recovery_rate = (
        final_recovered_amount / final_at_risk_amount
    ) * 100
else:
    final_recovery_rate = 0

print(f"At-Risk Payment Amount: ₹{final_at_risk_amount:,.2f}")
print(f"Final Recovered Amount: ₹{final_recovered_amount:,.2f}")
print(f"Final Unrecovered Amount: ₹{final_unrecovered_amount:,.2f}")
print(f"Final Recovery Rate: {final_recovery_rate:.1f}%")
print(f"Final Outcome: {final_status}")

print("\n✅ Final recovery outcome calculated successfully.")


========== FINAL RECOVERY OUTCOME ==========
At-Risk Payment Amount: ₹7,974.00
Final Recovered Amount: ₹7,974.00
Final Unrecovered Amount: ₹0.00
Final Recovery Rate: 100.0%
Final Outcome: RECOVERED

✅ Final recovery outcome calculated successfully.


In [15]:
# ============================================================
# FINAL AUDIT LOG
# CORRECTED VERSION
# ============================================================

print("========== FINAL AUDIT LOG ==========")

# ------------------------------------------------------------
# Get transaction ID safely
# ------------------------------------------------------------

if "transaction_id" in failed_payment.columns:

    actual_transaction_id = (
        failed_payment["transaction_id"].iloc[0]
    )

else:

    actual_transaction_id = (
        f"TX{failed_payment.index[0] + 1:05d}"
    )


# ------------------------------------------------------------
# Create final audit record
# ------------------------------------------------------------

final_audit_record = {

    "Transaction ID": actual_transaction_id,

    "Payment Amount (₹)": recovery_amount,

    "Actual Status": "FAILED",

    "Failure Probability (%)": round(
        failure_probability,
        2
    ),

    "Risk Level": risk_level,

    "Payment Error": failed_error,

    "First Recovery Action": recovery_action,

    "First Recovery Result": (
        "SUCCESS"
        if recovery_success
        else "FAILED"
    ),

    "Second Recovery Action": (
        second_recovery_action
        if stopping_decision
        == "CONTINUE - TRY ANOTHER RECOVERY ACTION"
        else "N/A"
    ),

    "Second Recovery Result": (
        "SUCCESS"
        if second_recovery_success
        else "FAILED"
    ),

    "Final Recovered Amount (₹)": (
        final_recovered_amount
    ),

    "Final Unrecovered Amount (₹)": (
        final_unrecovered_amount
    ),

    "Final Recovery Rate (%)": (
        round(final_recovery_rate, 1)
    ),

    "Final Outcome": final_status
}


# ------------------------------------------------------------
# Create final audit DataFrame
# ------------------------------------------------------------

final_audit_log = pd.DataFrame(
    [final_audit_record]
)


# ------------------------------------------------------------
# Display final audit log
# ------------------------------------------------------------

display(final_audit_log)

print("\n✅ Final audit log created successfully.")

========== FINAL AUDIT LOG ==========


,Transaction ID,Payment Amount (₹),Actual Status,Failure Probability (%),Risk Level,Payment Error,First Recovery Action,First Recovery Result,Second Recovery Action,Second Recovery Result,Final Recovered Amount (₹),Final Unrecovered Amount (₹),Final Recovery Rate (%),Final Outcome
0,TX16043,7974,FAILED,92.5,HIGH,AUTH_ERROR,Complete Authentication & Retry,FAILED,Alternative Payment Method,SUCCESS,7974,0,100.0,RECOVERED



✅ Final audit log created successfully.


In [16]:
# ============================================================
# LOAD TRAINED MODEL FOR VALIDATION
# ============================================================

import joblib

print("========== LOADING TRAINED MODEL ==========")

model = joblib.load("payment_model.pkl")

print("Model loaded successfully.")
print("Model classes:", model.classes_)
print("Ready for multi-transaction validation.")

========== LOADING TRAINED MODEL ==========
Model loaded successfully.
Model classes: ['FAILED' 'SUCCESS']
Ready for multi-transaction validation.


In [17]:
# ============================================================
# MULTI-TRANSACTION VALIDATION
# ADD BEFORE ORIGINAL CELL 4
# ============================================================

print("========== MULTI-TRANSACTION VALIDATION ==========")

# Select 10 real FAILED transactions
validation_transactions = df[
    df["status"] == "FAILED"
].head(10).copy()

print(
    f"Testing {len(validation_transactions)} "
    "real FAILED transactions from the dataset.\n"
)

# ------------------------------------------------------------
# Prepare features for the trained model
# ------------------------------------------------------------

validation_features = validation_transactions[
    [
        "amount",
        "payment_method",
        "bank",
        "device",
        "hour",
        "previous_failures",
        "attempt_number",
        "error_code",
        "customer_type"
    ]
]

# ------------------------------------------------------------
# AI prediction
# ------------------------------------------------------------

validation_probabilities = model.predict_proba(
    validation_features
)

failed_index = list(
    model.classes_
).index("FAILED")

validation_transactions[
    "AI Failure Probability (%)"
] = (
    validation_probabilities[:, failed_index] * 100
).round(2)

# ------------------------------------------------------------
# Determine AI risk level
# ------------------------------------------------------------

def get_risk_level(probability):

    if probability < 30:
        return "LOW"

    elif probability < 70:
        return "MEDIUM"

    else:
        return "HIGH"


validation_transactions[
    "AI Risk Level"
] = validation_transactions[
    "AI Failure Probability (%)"
].apply(get_risk_level)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

display(
    validation_transactions[
        [
            "transaction_id",
            "amount",
            "status",
            "error_code",
            "previous_failures",
            "attempt_number",
            "AI Failure Probability (%)",
            "AI Risk Level"
        ]
    ]
)

print(
    "\n✅ Multi-transaction AI validation completed."
)

========== MULTI-TRANSACTION VALIDATION ==========
Testing 10 real FAILED transactions from the dataset.



,transaction_id,amount,status,error_code,previous_failures,attempt_number,AI Failure Probability (%),AI Risk Level
6,TX00007,16950,FAILED,NONE,1,2,72.0,HIGH
8,TX00009,14523,FAILED,NONE,0,1,74.0,HIGH
17,TX00018,6520,FAILED,NONE,0,1,77.5,HIGH
18,TX00019,17668,FAILED,NONE,0,1,76.5,HIGH
19,TX00020,19869,FAILED,NETWORK_ERROR,0,1,36.0,MEDIUM
21,TX00022,8766,FAILED,NONE,1,2,19.5,LOW
23,TX00024,18531,FAILED,AUTH_ERROR,0,1,75.0,HIGH
26,TX00027,19218,FAILED,NETWORK_ERROR,2,3,93.0,HIGH
27,TX00028,3105,FAILED,BANK_ERROR,0,1,72.5,HIGH
28,TX00029,1999,FAILED,NONE,3,4,88.0,HIGH



✅ Multi-transaction AI validation completed.


In [18]:
# ============================================================
# AT-RISK PAYMENT VALUE VALIDATION
# ADD AFTER MULTI-TRANSACTION VALIDATION
# BEFORE ORIGINAL CELL 4
# ============================================================

print("========== AT-RISK PAYMENT VALUE VALIDATION ==========")

# ------------------------------------------------------------
# Total failed payment value in the validation sample
# ------------------------------------------------------------

total_failed_amount = validation_transactions[
    "amount"
].sum()


# ------------------------------------------------------------
# HIGH + MEDIUM = AI IDENTIFIED AT-RISK
# ------------------------------------------------------------

at_risk_transactions = validation_transactions[
    validation_transactions["AI Risk Level"].isin(
        ["HIGH", "MEDIUM"]
    )
]

at_risk_amount = at_risk_transactions[
    "amount"
].sum()


# ------------------------------------------------------------
# LOW-RISK FAILED TRANSACTIONS
# ------------------------------------------------------------

low_risk_transactions = validation_transactions[
    validation_transactions["AI Risk Level"] == "LOW"
]

low_risk_amount = low_risk_transactions[
    "amount"
].sum()


# ------------------------------------------------------------
# AT-RISK PERCENTAGE
# ------------------------------------------------------------

if total_failed_amount > 0:

    at_risk_percentage = (
        at_risk_amount
        / total_failed_amount
    ) * 100

else:

    at_risk_percentage = 0


# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------

print(
    f"Total Failed Payment Value: "
    f"₹{total_failed_amount:,.2f}"
)

print(
    f"AI Identified At-Risk Value: "
    f"₹{at_risk_amount:,.2f}"
)

print(
    f"Low-Risk Failed Value: "
    f"₹{low_risk_amount:,.2f}"
)

print(
    f"At-Risk Value Percentage: "
    f"{at_risk_percentage:.2f}%"
)

print(
    f"\nAt-Risk Transactions: "
    f"{len(at_risk_transactions)}"
)

print(
    f"Low-Risk Failed Transactions: "
    f"{len(low_risk_transactions)}"
)

print(
    "\n✅ At-risk payment value validation completed."
)

========== AT-RISK PAYMENT VALUE VALIDATION ==========
Total Failed Payment Value: ₹127,149.00
AI Identified At-Risk Value: ₹118,383.00
Low-Risk Failed Value: ₹8,766.00
At-Risk Value Percentage: 93.11%

At-Risk Transactions: 9
Low-Risk Failed Transactions: 1

✅ At-risk payment value validation completed.


In [19]:
# ============================================================
# MULTI-TRANSACTION RECOVERY SIMULATION
# ADD AFTER AT-RISK PAYMENT VALUE VALIDATION
# BEFORE ORIGINAL CELL 4
# ============================================================

print("========== MULTI-TRANSACTION RECOVERY SIMULATION ==========")

# ------------------------------------------------------------
# Work only with AI-identified at-risk transactions
# ------------------------------------------------------------

recovery_test = at_risk_transactions.copy()


# ------------------------------------------------------------
# Recovery probability based on AI risk
# ------------------------------------------------------------

def get_recovery_probability(risk_level):

    if risk_level == "HIGH":
        return 0.50

    elif risk_level == "MEDIUM":
        return 0.70

    return 0.90


# ------------------------------------------------------------
# Simulate recovery for each at-risk transaction
# ------------------------------------------------------------

recovery_results = []

for _, transaction in recovery_test.iterrows():

    risk = transaction["AI Risk Level"]

    amount = transaction["amount"]

    probability = get_recovery_probability(risk)

    recovery_success = (
        random.random() < probability
    )

    if recovery_success:

        recovered_amount = amount

        unrecovered_amount = 0

    else:

        recovered_amount = 0

        unrecovered_amount = amount

    recovery_results.append({

        "Transaction ID":
            transaction["transaction_id"],

        "Amount (₹)":
            amount,

        "Risk Level":
            risk,

        "AI Failure Probability (%)":
            transaction[
                "AI Failure Probability (%)"
            ],

        "Recovery Probability (%)":
            probability * 100,

        "Recovery Result":
            "RECOVERED"
            if recovery_success
            else "FAILED",

        "Recovered Amount (₹)":
            recovered_amount,

        "Unrecovered Amount (₹)":
            unrecovered_amount

    })


# ------------------------------------------------------------
# Create recovery results table
# ------------------------------------------------------------

recovery_results_df = pd.DataFrame(
    recovery_results,
    columns=[
        "Transaction ID",
        "Amount (₹)",
        "Risk Level",
        "AI Failure Probability (%)",
        "Recovery Probability (%)",
        "Recovery Result",
        "Recovered Amount (₹)",
        "Unrecovered Amount (₹)"
    ]
)


# ------------------------------------------------------------
# Calculate aggregate recovery metrics
# ------------------------------------------------------------

multi_total_at_risk = recovery_results_df[
    "Amount (₹)"
].sum()

multi_total_recovered = recovery_results_df[
    "Recovered Amount (₹)"
].sum()

multi_total_unrecovered = recovery_results_df[
    "Unrecovered Amount (₹)"
].sum()

multi_recovery_attempts = len(
    recovery_results_df
)

multi_successful_recoveries = (
    recovery_results_df[
        "Recovery Result"
    ] == "RECOVERED"
).sum()


if multi_recovery_attempts > 0:

    multi_recovery_rate = (
        multi_successful_recoveries
        / multi_recovery_attempts
    ) * 100

else:

    multi_recovery_rate = 0


# ------------------------------------------------------------
# Display transaction-level results
# ------------------------------------------------------------

display(
    recovery_results_df
)


# ------------------------------------------------------------
# Display business metrics
# ------------------------------------------------------------

print("\n========== RECOVERY METRICS ==========")

print(
    f"Total At-Risk Amount: "
    f"₹{multi_total_at_risk:,.2f}"
)

print(
    f"Total Recovered Amount: "
    f"₹{multi_total_recovered:,.2f}"
)

print(
    f"Total Unrecovered Amount: "
    f"₹{multi_total_unrecovered:,.2f}"
)

print(
    f"Recovery Attempts: "
    f"{multi_recovery_attempts}"
)

print(
    f"Successful Recoveries: "
    f"{multi_successful_recoveries}"
)

print(
    f"Recovery Success Rate: "
    f"{multi_recovery_rate:.1f}%"
)

print(
    "\n✅ Multi-transaction recovery simulation completed."
)

========== MULTI-TRANSACTION RECOVERY SIMULATION ==========


,Transaction ID,Amount (₹),Risk Level,AI Failure Probability (%),Recovery Probability (%),Recovery Result,Recovered Amount (₹),Unrecovered Amount (₹)
0,TX00007,16950,HIGH,72.0,50.0,RECOVERED,16950,0
1,TX00009,14523,HIGH,74.0,50.0,FAILED,0,14523
2,TX00018,6520,HIGH,77.5,50.0,RECOVERED,6520,0
3,TX00019,17668,HIGH,76.5,50.0,RECOVERED,17668,0
4,TX00020,19869,MEDIUM,36.0,70.0,RECOVERED,19869,0
5,TX00024,18531,HIGH,75.0,50.0,FAILED,0,18531
6,TX00027,19218,HIGH,93.0,50.0,FAILED,0,19218
7,TX00028,3105,HIGH,72.5,50.0,RECOVERED,3105,0
8,TX00029,1999,HIGH,88.0,50.0,FAILED,0,1999



========== RECOVERY METRICS ==========
Total At-Risk Amount: ₹118,383.00
Total Recovered Amount: ₹64,112.00
Total Unrecovered Amount: ₹54,271.00
Recovery Attempts: 9
Successful Recoveries: 5
Recovery Success Rate: 55.6%

✅ Multi-transaction recovery simulation completed.


In [20]:
# ============================================================
# FINAL BUSINESS IMPACT SUMMARY
# ADD AFTER MULTI-TRANSACTION RECOVERY SIMULATION
# ============================================================

print("========== FINAL BUSINESS IMPACT SUMMARY ==========")

# ------------------------------------------------------------
# Total failed payment value in validation sample
# ------------------------------------------------------------

final_failed_value = total_failed_amount


# ------------------------------------------------------------
# AI identified at-risk payment value
# ------------------------------------------------------------

final_at_risk_value = at_risk_amount


# ------------------------------------------------------------
# Recovery results
# ------------------------------------------------------------

final_recovered_value = multi_total_recovered

final_unrecovered_value = multi_total_unrecovered


# ------------------------------------------------------------
# Percentage of failed value identified as at-risk
# ------------------------------------------------------------

if final_failed_value > 0:

    at_risk_value_percentage = (
        final_at_risk_value
        / final_failed_value
    ) * 100

else:

    at_risk_value_percentage = 0


# ------------------------------------------------------------
# Percentage of at-risk value recovered
# ------------------------------------------------------------

if final_at_risk_value > 0:

    value_recovery_percentage = (
        final_recovered_value
        / final_at_risk_value
    ) * 100

else:

    value_recovery_percentage = 0


# ------------------------------------------------------------
# Percentage of total failed value recovered
# ------------------------------------------------------------

if final_failed_value > 0:

    overall_value_recovery = (
        final_recovered_value
        / final_failed_value
    ) * 100

else:

    overall_value_recovery = 0


# ------------------------------------------------------------
# DISPLAY FINAL BUSINESS RESULTS
# ------------------------------------------------------------

print(
    f"Total Failed Payment Value: "
    f"₹{final_failed_value:,.2f}"
)

print(
    f"AI Identified At-Risk Value: "
    f"₹{final_at_risk_value:,.2f}"
)

print(
    f"At-Risk Value Identified: "
    f"{at_risk_value_percentage:.2f}%"
)

print(
    f"Recovered Payment Value: "
    f"₹{final_recovered_value:,.2f}"
)

print(
    f"Unrecovered Payment Value: "
    f"₹{final_unrecovered_value:,.2f}"
)

print(
    f"Recovery Rate by At-Risk Value: "
    f"{value_recovery_percentage:.2f}%"
)

print(
    f"Overall Failed-Value Recovery: "
    f"{overall_value_recovery:.2f}%"
)

print(
    "\n✅ Final business impact calculated successfully."
)

========== FINAL BUSINESS IMPACT SUMMARY ==========
Total Failed Payment Value: ₹127,149.00
AI Identified At-Risk Value: ₹118,383.00
At-Risk Value Identified: 93.11%
Recovered Payment Value: ₹64,112.00
Unrecovered Payment Value: ₹54,271.00
Recovery Rate by At-Risk Value: 54.16%
Overall Failed-Value Recovery: 50.42%

✅ Final business impact calculated successfully.


In [ ]:
# ============================================================
# FINAL TRANSACTION-LEVEL RECOVERY VALIDATION
# ============================================================

print("========== TRANSACTION-LEVEL RECOVERY VALIDATION ==========")

# Use the ACTUAL recovery DataFrame from our notebook
final_recovery_df = recovery_results_df.copy()

# ------------------------------------------------------------
# FINAL RECOVERED AMOUNT
# Each transaction amount can be recovered ONLY ONCE
# ------------------------------------------------------------

final_recovery_df["Final Recovered Amount (₹)"] = (
    final_recovery_df["Recovered Amount (₹)"]
)

# ------------------------------------------------------------
# FINAL UNRECOVERED AMOUNT
# ------------------------------------------------------------

final_recovery_df["Final Unrecovered Amount (₹)"] = (
    final_recovery_df["Unrecovered Amount (₹)"]
)

# ------------------------------------------------------------
# BUSINESS TOTALS
# ------------------------------------------------------------

final_at_risk_amount = (
    final_recovery_df["Amount (₹)"].sum()
)

final_recovered_amount = (
    final_recovery_df["Final Recovered Amount (₹)"].sum()
)

final_unrecovered_amount = (
    final_recovery_df["Final Unrecovered Amount (₹)"].sum()
)

# ------------------------------------------------------------
# RECOVERY RATE BY MONEY VALUE
# ------------------------------------------------------------

if final_at_risk_amount > 0:

    final_recovery_rate = (
        final_recovered_amount
        / final_at_risk_amount
    ) * 100

else:

    final_recovery_rate = 0

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

print(f"Total At-Risk Amount: ₹{final_at_risk_amount:,.2f}")

print(
    f"Final Recovered Amount: "
    f"₹{final_recovered_amount:,.2f}"
)

print(
    f"Final Unrecovered Amount: "
    f"₹{final_unrecovered_amount:,.2f}"
)

print(
    f"Final Recovery Rate: "
    f"{final_recovery_rate:.2f}%"
)

print()
print("Business rule:")
print("Each transaction amount is counted only once.")

print()
print("✅ Final transaction-level recovery validation completed.")


In [ ]:
# ============================================================
# FINAL MULTI-TRANSACTION AUDIT LOG
# ============================================================

print("========== FINAL MULTI-TRANSACTION AUDIT LOG ==========")

# Build an auditable record for every AI-selected recovery case.
multi_audit_log = final_recovery_df[
    [
        "Transaction ID",
        "Amount (₹)",
        "Risk Level",
        "AI Failure Probability (%)",
        "Recovery Probability (%)",
        "Recovery Result",
        "Final Recovered Amount (₹)",
        "Final Unrecovered Amount (₹)"
    ]
].copy()

multi_audit_log["Recovery Attempts"] = 1

multi_audit_log["Stopping Rule"] = multi_audit_log[
    "Recovery Result"
].apply(
    lambda result:
        "STOP - PAYMENT RECOVERED"
        if result == "RECOVERED"
        else "CONTINUE - ANOTHER RECOVERY ACTION ALLOWED"
)

display(multi_audit_log)

# Save the audit trail for the future dashboard/backend.
multi_audit_log.to_csv(
    "payment_recovery_audit_log.csv",
    index=False
)

print("\nAudit records:", len(multi_audit_log))
print("✅ Final multi-transaction audit log created.")
print("File: payment_recovery_audit_log.csv")


In [ ]:
# ============================================================
# MULTI-TRANSACTION RECOVERY STOPPING RULE
# ============================================================

print("========== MULTI-TRANSACTION STOPPING RULE ==========")

MAX_RECOVERY_ATTEMPTS = 2
CURRENT_RECOVERY_ATTEMPT = 1

print(
    f"Maximum Allowed Recovery Attempts: "
    f"{MAX_RECOVERY_ATTEMPTS}"
)
print(
    f"Current Recovery Attempt: "
    f"{CURRENT_RECOVERY_ATTEMPT}"
)

# In this prototype, the multi-transaction simulation performs
# one recovery attempt. A failed recovery remains eligible for
# one additional attempt, while a successful recovery stops.
if CURRENT_RECOVERY_ATTEMPT < MAX_RECOVERY_ATTEMPTS:
    stopping_decision = (
        "CONTINUE - FAILED PAYMENTS MAY RECEIVE ONE MORE ACTION"
    )
else:
    stopping_decision = (
        "STOP - MAXIMUM RECOVERY ATTEMPTS REACHED"
    )

print(f"\nStopping Rule Decision: {stopping_decision}")

print("\nBusiness Rule:")
print(
    f"The system allows a maximum of "
    f"{MAX_RECOVERY_ATTEMPTS} recovery attempts per transaction."
)
print(
    "If a payment is recovered, further recovery attempts are stopped."
)

print("\n✅ Multi-transaction stopping rule evaluated successfully.")


In [ ]:
# ============================================================
# FINAL PROTOTYPE SUMMARY
# ============================================================

print("========== FINAL PAYMENT RECOVERY AI SUMMARY ==========")

print()
print("1. FAILED PAYMENTS ANALYZED")
print(f"   Total Failed Payment Value: ₹{final_recovery_df['Amount (₹)'].sum():,.2f}")

print()
print("2. AI RISK IDENTIFICATION")
print(f"   At-Risk Payment Value: ₹{final_at_risk_amount:,.2f}")

print()
print("3. RECOVERY PERFORMANCE")
print(f"   Recovered Payment Value: ₹{final_recovered_amount:,.2f}")
print(f"   Unrecovered Payment Value: ₹{final_unrecovered_amount:,.2f}")
print(f"   Recovery Rate: {final_recovery_rate:.2f}%")

print()
print("4. RECOVERY CONTROL")
print(f"   Maximum Recovery Attempts: {MAX_RECOVERY_ATTEMPTS}")
print(f"   Stopping Rule: ENABLED")

print()
print("5. AUDITABILITY")
print("   AI Decision: Logged")
print("   Recovery Action: Logged")
print("   Recovery Result: Logged")
print("   Recovered Amount: Logged")
print("   Unrecovered Amount: Logged")

print()
print("6. BUSINESS RULE")
print("   Each transaction amount is counted only once.")
print("   Multiple recovery attempts do not double-count recovered money.")

print()
print("========== PROTOTYPE CORE FLOW COMPLETE ==========")
print("FAILED PAYMENT")
print("      ↓")
print("AI RISK PREDICTION")
print("      ↓")
print("RECOVERY ACTION")
print("      ↓")
print("RECOVERY SIMULATION")
print("      ↓")
print("RECOVERED / UNRECOVERED")
print("      ↓")
print("BUSINESS METRICS")
print("      ↓")
print("AUDIT LOG")
print("      ↓")
print("STOPPING RULE")

print()
print("✅ FINAL PROTOTYPE SUMMARY GENERATED SUCCESSFULLY.")

In [ ]:
# ============================================================
# CORRECT FINAL BUSINESS IMPACT SUMMARY
# ============================================================

print("========== CORRECT FINAL BUSINESS IMPACT SUMMARY ==========")

# ------------------------------------------------------------
# TOTAL FAILED PAYMENT VALUE
# From all 10 FAILED transactions used for validation
# ------------------------------------------------------------

total_failed_value = validation_transactions[
    validation_transactions["status"] == "FAILED"
]["amount"].sum()

# ------------------------------------------------------------
# AI IDENTIFIED AT-RISK VALUE
# Only MEDIUM + HIGH risk transactions
# ------------------------------------------------------------

ai_at_risk_value = at_risk_transactions["amount"].sum()

# ------------------------------------------------------------
# LOW-RISK FAILED VALUE
# ------------------------------------------------------------

low_risk_failed_value = low_risk_transactions["amount"].sum()

# ------------------------------------------------------------
# RECOVERY RESULTS
# ------------------------------------------------------------

recovered_value = final_recovered_amount

unrecovered_value = final_unrecovered_amount

# ------------------------------------------------------------
# PERCENTAGES
# ------------------------------------------------------------

if total_failed_value > 0:
    at_risk_percentage = (
        ai_at_risk_value / total_failed_value
    ) * 100

    overall_failed_recovery = (
        recovered_value / total_failed_value
    ) * 100
else:
    at_risk_percentage = 0
    overall_failed_recovery = 0

if ai_at_risk_value > 0:
    recovery_rate_by_risk = (
        recovered_value / ai_at_risk_value
    ) * 100
else:
    recovery_rate_by_risk = 0

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

print(f"Total Failed Payment Value: ₹{total_failed_value:,.2f}")

print(
    f"AI Identified At-Risk Value: "
    f"₹{ai_at_risk_value:,.2f}"
)

print(
    f"Low-Risk Failed Value: "
    f"₹{low_risk_failed_value:,.2f}"
)

print(
    f"At-Risk Value Identified: "
    f"{at_risk_percentage:.2f}%"
)

print(
    f"Recovered Payment Value: "
    f"₹{recovered_value:,.2f}"
)

print(
    f"Unrecovered Payment Value: "
    f"₹{unrecovered_value:,.2f}"
)

print(
    f"Recovery Rate by At-Risk Value: "
    f"{recovery_rate_by_risk:.2f}%"
)

print(
    f"Overall Failed-Value Recovery: "
    f"{overall_failed_recovery:.2f}%"
)

print()
print("Business rule:")
print("Only MEDIUM/HIGH risk payments enter recovery.")
print("Each transaction amount is counted only once.")

print()
print("✅ Correct final business impact calculated successfully.")

In [ ]:
# ============================================================
# SAVE FINAL PROTOTYPE ARTIFACTS
# ============================================================

import joblib
import os

print("========== FINAL ARTIFACTS ==========")

if "pipeline" in globals():
    joblib.dump(pipeline, "payment_model.pkl")
    print("payment_model.pkl saved")

if "final_recovery_df" in globals():
    final_recovery_df.to_csv("payment_recovery_results.csv", index=False)
    print("payment_recovery_results.csv saved")

if "multi_audit_log" in globals():
    multi_audit_log.to_csv("payment_recovery_audit_log.csv", index=False)
    print("payment_recovery_audit_log.csv saved")

print("Final artifact-saving step completed.")
